# EcoSignal — Features 18 and 19 Real-Image Runner

This notebook performs the remaining real-input work for:

- **Feature 18:** run the finalized coverage segmentation model on two confirmed repeat photographs, create `repeat_scan_manifest.csv`, align the photographs, and report an uncertainty-aware coverage change.
- **Feature 19:** run image-quality and lichen-framing checks on a real photograph and save an annotated overlay plus frontend-ready JSON.

Scientific safeguards:

- No synthetic observations are created.
- The same-site relationship must be explicitly confirmed.
- Unknown dates remain blank and are explicitly recorded as unavailable.
- A coverage change is not reported when the images are insufficiently comparable.
- The interval uses the finalized model's held-out coverage MAE of **3.9827 percentage points**.

This notebook intentionally performs **no package upgrades**. It does not use Gemini, OpenAI, Numba, or `google.generativeai`.

In [ ]:
# Cell 1 — Safe imports and runtime check (do not add a pip install above this cell)
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
import json, math, platform

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageDraw
from IPython.display import display

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("Runtime imports: PASS")

## Put the files in Google Drive

Create these folders:

```text
MyDrive/EcoSignal/models/
MyDrive/EcoSignal/repeat_scans/
MyDrive/EcoSignal/guided_capture/
```

Upload your model as `models/lichen_coverage_model.keras`. Put the **older** and **latest** repeat photographs in `repeat_scans`, and put the Feature 19 photograph in `guided_capture`.

Then edit only the configuration values in the next cell. If the original capture dates are unknown, leave both date fields blank. Do not use screenshot-export dates or invent calendar dates. The notebook can still make an explicitly ordered **older-versus-latest pairwise comparison**, but it will disable elapsed-time, rate, and trend claims.

In [ ]:
# Cell 2 — Mount Drive and EDIT THIS CONFIGURATION
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/EcoSignal')
MODEL_PATH = ROOT / 'models/lichen_coverage_model.keras'

# Feature 18: same lichen/tree, older image first and latest image second.
BASELINE_IMAGE = ROOT / 'repeat_scans/older.png'
CURRENT_IMAGE = ROOT / 'repeat_scans/latest.png'

# Feature 19: the photo to test with Guided Capture.
GUIDED_CAPTURE_IMAGE = ROOT / 'guided_capture/feature19.jpeg'

# REQUIRED REAL METADATA.
MONITORING_SITE_ID = 'TREE-001'
IMAGE_ORDER_CONFIRMED = True  # True only because you know which image is older/latest.

# Optional. Leave BOTH blank when the original capture dates are unknown.
# ISO-8601 example format: 2026-05-20T12:00:00-04:00
BASELINE_CAPTURED_AT = ''
CURRENT_CAPTURED_AT = ''

# Optional, but add them if known. Leave as None rather than guessing.
LATITUDE = None
LONGITUDE = None
SUBSTRATE_TYPE = 'tree_bark'
BASELINE_NOTES = 'Original capture date unknown; image confirmed as older.'
CURRENT_NOTES = 'Original capture date unknown; image confirmed as latest.'

OUTPUT_DIR = ROOT / 'features_18_19_real_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COVERAGE_THRESHOLD = 0.70
COVERAGE_HELDOUT_MAE_PP = 3.9827008928571437

print('Output folder:', OUTPUT_DIR)

In [ ]:
# Cell 3 — Validate files and metadata before running either feature
required_files = {
    'coverage model': MODEL_PATH,
    'Feature 18 older image': BASELINE_IMAGE,
    'Feature 18 latest image': CURRENT_IMAGE,
    'Feature 19 image': GUIDED_CAPTURE_IMAGE,
}
missing = [f'{label}: {path}' for label, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

if not MONITORING_SITE_ID.strip():
    raise ValueError('Set MONITORING_SITE_ID to a stable ID for this exact tree/site.')
if not IMAGE_ORDER_CONFIRMED:
    raise ValueError('Confirm which photograph is older and which is latest before comparing them.')
if bool(BASELINE_CAPTURED_AT) != bool(CURRENT_CAPTURED_AT):
    raise ValueError('Provide both capture dates or leave both blank; do not provide only one.')

if BASELINE_CAPTURED_AT and CURRENT_CAPTURED_AT:
    baseline_dt = pd.to_datetime(BASELINE_CAPTURED_AT, utc=True, errors='raise')
    current_dt = pd.to_datetime(CURRENT_CAPTURED_AT, utc=True, errors='raise')
    if current_dt <= baseline_dt:
        raise ValueError('CURRENT_CAPTURED_AT must be later than BASELINE_CAPTURED_AT.')
    temporal_metadata_status = 'dates_known'
    print('Time between repeat scans:', current_dt - baseline_dt)
else:
    baseline_dt = current_dt = None
    temporal_metadata_status = 'dates_unknown_order_only'
    print('Capture dates: UNKNOWN. Pairwise ordering only; elapsed-time and trend claims are disabled.')
if BASELINE_IMAGE.resolve() == CURRENT_IMAGE.resolve():
    raise ValueError('The baseline and current paths point to the same file.')

print('Input validation: PASS')

In [ ]:
# Cell 4 — Load the finalized coverage model and verify its input contract
coverage_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print('Model input shape:', coverage_model.input_shape)
print('Model output shape:', coverage_model.output_shape)

# The supplied MobileNetV2 U-Net contains a Rescaling(scale=2, offset=-1) layer.
# Therefore inputs must remain RGB float32 values in [0,255]; dividing by 255 here
# would apply normalization twice and produce invalid results.
rescaling_layers = [layer for layer in coverage_model.layers if isinstance(layer, tf.keras.layers.Rescaling)]
if not rescaling_layers:
    # The Rescaling layer can be nested inside an encoder model.
    for layer in coverage_model.layers:
        if isinstance(layer, tf.keras.Model):
            rescaling_layers.extend(x for x in layer.layers if isinstance(x, tf.keras.layers.Rescaling))

if rescaling_layers:
    layer = rescaling_layers[0]
    print('Internal rescaling detected:', {'scale': float(layer.scale), 'offset': float(layer.offset)})
    MODEL_EXPECTS_RAW_255 = True
else:
    print('No internal Rescaling layer detected; inputs will be scaled to [0,1].')
    MODEL_EXPECTS_RAW_255 = False

assert len(coverage_model.input_shape) == 4 and coverage_model.input_shape[-1] == 3
print('Coverage model connection: PASS')

In [ ]:
# Cell 5 — Correct coverage inference, mask cleanup, confidence heuristic, and saved overlays
def load_rgb(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert('RGB'))

def predict_coverage(path: Path, threshold: float = COVERAGE_THRESHOLD) -> dict[str, Any]:
    image = load_rgb(path)
    input_h = int(coverage_model.input_shape[1])
    input_w = int(coverage_model.input_shape[2])
    resized = cv2.resize(image, (input_w, input_h), interpolation=cv2.INTER_AREA).astype(np.float32)
    model_input = resized if MODEL_EXPECTS_RAW_255 else resized / 255.0
    probability_small = np.asarray(coverage_model.predict(model_input[None, ...], verbose=0)).squeeze()
    if probability_small.ndim == 3:
        probability_small = probability_small[..., 0]
    if probability_small.ndim != 2:
        raise ValueError(f'Expected a single-channel segmentation output; got {probability_small.shape}')

    probability = cv2.resize(
        probability_small.astype(np.float32),
        (image.shape[1], image.shape[0]),
        interpolation=cv2.INTER_LINEAR,
    ).clip(0, 1)
    mask = (probability >= threshold).astype(np.uint8)
    kernel_size = max(3, int(round(min(image.shape[:2]) / 300)))
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # This is an inference-certainty heuristic, not a calibrated probability.
    eps = 1e-7
    entropy = -(probability*np.log2(probability+eps) + (1-probability)*np.log2(1-probability+eps))
    confidence_heuristic = float(np.clip(1 - entropy.mean(), 0, 1))
    coverage_percent = float(100 * mask.mean())
    return {
        'image': image,
        'probability': probability,
        'mask': mask,
        'coverage_percent': coverage_percent,
        'coverage_confidence': confidence_heuristic,
        'confidence_method': 'mean_binary_entropy_heuristic_not_calibrated',
    }

def save_coverage_artifacts(source_path: Path, result: dict[str, Any], prefix: str) -> dict[str, str]:
    mask_path = OUTPUT_DIR / f'{prefix}_coverage_mask.png'
    overlay_path = OUTPUT_DIR / f'{prefix}_coverage_overlay.png'
    probability_path = OUTPUT_DIR / f'{prefix}_coverage_probability.png'
    Image.fromarray((result['mask'] * 255).astype(np.uint8)).save(mask_path)
    Image.fromarray((result['probability'] * 255).astype(np.uint8)).save(probability_path)
    image = result['image'].astype(np.float32)
    green = np.zeros_like(image); green[..., 1] = 255
    alpha = (0.42 * result['mask'])[..., None]
    overlay = (image*(1-alpha) + green*alpha).clip(0,255).astype(np.uint8)
    Image.fromarray(overlay).save(overlay_path)
    return {'mask_path': str(mask_path), 'overlay_path': str(overlay_path),
            'probability_path': str(probability_path)}

baseline_coverage = predict_coverage(BASELINE_IMAGE)
current_coverage = predict_coverage(CURRENT_IMAGE)
baseline_artifacts = save_coverage_artifacts(BASELINE_IMAGE, baseline_coverage, 'feature18_baseline')
current_artifacts = save_coverage_artifacts(CURRENT_IMAGE, current_coverage, 'feature18_current')

print(f"Older coverage: {baseline_coverage['coverage_percent']:.2f}%")
print(f"Latest coverage: {current_coverage['coverage_percent']:.2f}%")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].imshow(baseline_coverage['image']); axes[0,0].set_title('Older original')
axes[0,1].imshow(Image.open(baseline_artifacts['overlay_path'])); axes[0,1].set_title(f"Older mask: {baseline_coverage['coverage_percent']:.2f}%")
axes[1,0].imshow(current_coverage['image']); axes[1,0].set_title('Latest original')
axes[1,1].imshow(Image.open(current_artifacts['overlay_path'])); axes[1,1].set_title(f"Latest mask: {current_coverage['coverage_percent']:.2f}%")
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 6 — Create the exact Feature 18 repeat_scan_manifest.csv automatically
REPEAT_COLUMNS = [
    'sample_id','monitoring_site_id','site_id_confirmed','site_match_method','site_match_distance_m','captured_at','image_path',
    'latitude','longitude','substrate_type','coverage_percent','coverage_confidence',
    'growth_form','growth_form_confidence','aqi','notes','capture_sequence','temporal_order_basis'
]

manifest = pd.DataFrame([
    {
        'sample_id': f'{MONITORING_SITE_ID}-baseline',
        'monitoring_site_id': MONITORING_SITE_ID,
        'site_id_confirmed': True,
        'site_match_method': 'user_confirmed_same_tree',
        'site_match_distance_m': 0.0,
        'captured_at': BASELINE_CAPTURED_AT,
        'image_path': str(BASELINE_IMAGE),
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'substrate_type': SUBSTRATE_TYPE,
        'coverage_percent': round(baseline_coverage['coverage_percent'], 4),
        'coverage_confidence': round(baseline_coverage['coverage_confidence'], 4),
        'growth_form': None,
        'growth_form_confidence': None,
        'aqi': None,
        'notes': BASELINE_NOTES,
        'capture_sequence': 1,
        'temporal_order_basis': 'user_confirmed_older_vs_latest',
    },
    {
        'sample_id': f'{MONITORING_SITE_ID}-current',
        'monitoring_site_id': MONITORING_SITE_ID,
        'site_id_confirmed': True,
        'site_match_method': 'user_confirmed_same_tree',
        'site_match_distance_m': 0.0,
        'captured_at': CURRENT_CAPTURED_AT,
        'image_path': str(CURRENT_IMAGE),
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'substrate_type': SUBSTRATE_TYPE,
        'coverage_percent': round(current_coverage['coverage_percent'], 4),
        'coverage_confidence': round(current_coverage['coverage_confidence'], 4),
        'growth_form': None,
        'growth_form_confidence': None,
        'aqi': None,
        'notes': CURRENT_NOTES,
        'capture_sequence': 2,
        'temporal_order_basis': 'user_confirmed_older_vs_latest',
    },
], columns=REPEAT_COLUMNS)

MANIFEST_PATH = ROOT / 'repeat_scan_manifest.csv'
manifest.to_csv(MANIFEST_PATH, index=False)
display(manifest)
print('Saved Feature 18 manifest:', MANIFEST_PATH)

In [ ]:
# Cell 7 — Feature 18 geometric alignment and comparability
def load_cv_image(path: Path, max_dimension: int = 1400) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(path)
    h, w = image.shape[:2]
    scale = min(1.0, max_dimension / max(h, w))
    return cv2.resize(image, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_AREA) if scale < 1 else image

def normalized_luminance(image_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)

def sharpness(image_bgr: np.ndarray) -> float:
    return float(cv2.Laplacian(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())

def global_ssim(a: np.ndarray, b: np.ndarray) -> float:
    # Standard global SSIM formula; avoids adding another package to the Colab runtime.
    a, b = a.astype(np.float64), b.astype(np.float64)
    c1, c2 = (0.01*255)**2, (0.03*255)**2
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(), b.var()
    covariance = ((a-ma)*(b-mb)).mean()
    return float(((2*ma*mb+c1)*(2*covariance+c2))/((ma*ma+mb*mb+c1)*(va+vb+c2)))

def align_repeat_images(baseline_path: Path, current_path: Path) -> dict[str, Any]:
    base = load_cv_image(baseline_path)
    current = load_cv_image(current_path)
    current = cv2.resize(current, (base.shape[1], base.shape[0]), interpolation=cv2.INTER_AREA)
    bg, cg = normalized_luminance(base), normalized_luminance(current)
    orb = cv2.ORB_create(nfeatures=3500, scaleFactor=1.2, nlevels=8)
    key1, desc1 = orb.detectAndCompute(bg, None)
    key2, desc2 = orb.detectAndCompute(cg, None)
    if desc1 is None or desc2 is None or len(key1) < 12 or len(key2) < 12:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Insufficient stable visual features for same-site alignment.',
                'metrics':{'keypoints_baseline':len(key1),'keypoints_current':len(key2)}}
    pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(desc2, desc1, k=2)
    good = [m for m,n in pairs if m.distance < 0.72*n.distance]
    if len(good) < 12:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Too few consistent matches; reproduce the original framing.',
                'metrics':{'good_matches':len(good)}}
    src = np.float32([key2[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([key1[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    homography, inliers = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
    if homography is None or inliers is None:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'RANSAC could not establish stable alignment.', 'metrics':{'good_matches':len(good)}}
    aligned = cv2.warpPerspective(current, homography, (base.shape[1],base.shape[0]))
    valid = cv2.warpPerspective(np.full(current.shape[:2],255,np.uint8), homography,
                                (base.shape[1],base.shape[0])) > 0
    overlap = float(valid.mean())
    inlier_ratio = float(inliers.ravel().mean())
    ys, xs = np.where(valid)
    if len(xs) < 100:
        return {'decision':'retake_required','comparability_score':0.0,
                'reason':'Aligned overlap is too small.', 'metrics':{'overlap_fraction':overlap}}
    x0,x1,y0,y1 = xs.min(),xs.max()+1,ys.min(),ys.max()+1
    ssim = global_ssim(normalized_luminance(base)[y0:y1,x0:x1], normalized_luminance(aligned)[y0:y1,x0:x1])
    exposure_difference = abs(float(cv2.cvtColor(base,cv2.COLOR_BGR2GRAY).mean()) -
                              float(cv2.cvtColor(aligned,cv2.COLOR_BGR2GRAY)[valid].mean()))/255
    sharp_ratio = min(sharpness(base),sharpness(aligned))/max(sharpness(base),sharpness(aligned),1e-6)
    components = {
        'ransac_inlier_ratio':float(np.clip(inlier_ratio/0.65,0,1)),
        'structural_similarity':float(np.clip((ssim-0.15)/0.70,0,1)),
        'exposure_consistency':float(np.clip(1-exposure_difference/0.35,0,1)),
        'overlap_fraction':float(np.clip(overlap/0.85,0,1)),
        'sharpness_consistency':float(np.clip(sharp_ratio,0,1)),
    }
    score = 0.32*components['ransac_inlier_ratio'] + 0.30*components['structural_similarity'] + \
            0.16*components['exposure_consistency'] + 0.12*components['overlap_fraction'] + \
            0.10*components['sharpness_consistency']
    decision = 'retake_required' if overlap < 0.35 else ('accepted' if score >= 0.72 else ('caution' if score >= 0.52 else 'retake_required'))
    reason = {'accepted':'Images are sufficiently comparable for screening-level change analysis.',
              'caution':'Images can be compared only with increased uncertainty.',
              'retake_required':'Framing or visual conditions differ too much for a defensible comparison.'}[decision]
    return {'decision':decision,'comparability_score':round(float(score),4),'reason':reason,
            'metrics':{**{k:round(v,4) for k,v in components.items()},'raw_ssim':round(ssim,4),
                       'raw_inlier_ratio':round(inlier_ratio,4),'raw_overlap_fraction':round(overlap,4),
                       'exposure_difference':round(exposure_difference,4),'good_matches':len(good)},
            '_aligned_image':aligned,'_baseline_image':base}

alignment = align_repeat_images(BASELINE_IMAGE, CURRENT_IMAGE)
print(json.dumps({k:v for k,v in alignment.items() if not k.startswith('_')}, indent=2))
if '_aligned_image' in alignment:
    aligned_path = OUTPUT_DIR / 'feature18_current_aligned_to_baseline.png'
    cv2.imwrite(str(aligned_path), alignment['_aligned_image'])
    fig, ax = plt.subplots(1,2,figsize=(14,7))
    ax[0].imshow(cv2.cvtColor(alignment['_baseline_image'],cv2.COLOR_BGR2RGB)); ax[0].set_title('Older reference')
    ax[1].imshow(cv2.cvtColor(alignment['_aligned_image'],cv2.COLOR_BGR2RGB)); ax[1].set_title('Latest aligned to older')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 8 — Feature 18 uncertainty-aware change result
def coverage_change_interval(previous: float, current: float, comparability_score: float) -> dict[str, Any]:
    delta = float(current-previous)
    residual_sigma = COVERAGE_HELDOUT_MAE_PP * math.sqrt(math.pi/2)
    base_half_width = 1.96 * math.sqrt(2) * residual_sigma
    half_width = base_half_width / max(float(comparability_score),0.45)
    lower, upper = delta-half_width, delta+half_width
    status = 'increase_supported' if lower > 0 else ('decrease_supported' if upper < 0 else 'no_reliable_change_detected')
    return {'coverage_delta_pp':round(delta,2),'approx_95_interval_pp':[round(lower,2),round(upper,2)],
            'change_status':status,'mae_basis_pp':COVERAGE_HELDOUT_MAE_PP,
            'interval_method':'held-out-MAE normal approximation, comparability-inflated; screening only'}

if alignment['decision'] == 'retake_required':
    coverage_change = {'change_status':'unavailable','reason':alignment['reason']}
else:
    coverage_change = coverage_change_interval(baseline_coverage['coverage_percent'],
                                               current_coverage['coverage_percent'],
                                               alignment['comparability_score'])

repeat_result = {
    'feature':'repeat_scan_monitoring','version':'repeat_scan_v1.0.2_unknown_dates_supported',
    'monitoring_site_id':MONITORING_SITE_ID,
    'same_site_user_confirmed':True,
    'image_order_user_confirmed':IMAGE_ORDER_CONFIRMED,
    'temporal_metadata_status':temporal_metadata_status,
    'previous_sample_id':f'{MONITORING_SITE_ID}-baseline',
    'current_sample_id':f'{MONITORING_SITE_ID}-current',
    'previous_captured_at':BASELINE_CAPTURED_AT or None,
    'current_captured_at':CURRENT_CAPTURED_AT or None,
    'previous_coverage_percent':round(baseline_coverage['coverage_percent'],2),
    'current_coverage_percent':round(current_coverage['coverage_percent'],2),
    'comparability':{k:v for k,v in alignment.items() if not k.startswith('_')},
    'coverage_change':coverage_change,
    'elapsed_time_available':bool(BASELINE_CAPTURED_AT and CURRENT_CAPTURED_AT),
    'rate_or_trend_claim_allowed':False,
    'interpretation_boundary':'Ordered pairwise screening result only; unknown dates prevent elapsed-time or rate claims, and two observations do not establish an ecological trend or pollution cause.'
}
repeat_path = OUTPUT_DIR / 'repeat_scan_results.json'
repeat_path.write_text(json.dumps(repeat_result,indent=2))
print(json.dumps(repeat_result,indent=2))
print('Saved:',repeat_path)

In [ ]:
# Cell 9 — Feature 19 Guided Capture using the same correctly preprocessed coverage model
@dataclass(frozen=True)
class CaptureThresholds:
    min_short_side:int=384; min_total_pixels:int=600_000; min_blur_variance:float=85.0
    min_mean_luminance:float=0.18; max_mean_luminance:float=0.86; min_robust_contrast:float=0.20
    max_underexposed_fraction:float=0.22; max_overexposed_fraction:float=0.18
    min_lichen_fraction:float=0.08; max_lichen_fraction:float=0.92; max_centroid_offset:float=0.34
    max_border_touch_sides:int=3

def base_capture_metrics(image:np.ndarray)->dict[str,float]:
    h,w=image.shape[:2]; scale=min(1.0,900/max(h,w))
    stable=cv2.resize(image,(round(w*scale),round(h*scale)),interpolation=cv2.INTER_AREA) if scale<1 else image
    gray=cv2.cvtColor(stable,cv2.COLOR_RGB2GRAY); luma=gray.astype(np.float32)/255
    p05,p95=np.percentile(luma,[5,95]); hsv=cv2.cvtColor(stable,cv2.COLOR_RGB2HSV)
    return {'width':w,'height':h,'blur_variance':float(cv2.Laplacian(gray,cv2.CV_64F).var()),
            'mean_luminance':float(luma.mean()),'robust_contrast':float(p95-p05),
            'underexposed_fraction':float((luma<0.05).mean()),'overexposed_fraction':float((luma>0.98).mean()),
            'mean_saturation':float((hsv[...,1].astype(np.float32)/255).mean())}

def mask_framing_metrics(mask:np.ndarray)->dict[str,Any]:
    binary=(mask>0).astype(np.uint8); fraction=float(binary.mean())
    count,labels,stats,centroids=cv2.connectedComponentsWithStats(binary,8)
    if count<=1 or binary.sum()==0:
        return {'lichen_fraction':fraction,'largest_component_fraction':0.0,'centroid_offset':1.0,'border_touch_sides':0}
    component=1+int(np.argmax(stats[1:,cv2.CC_STAT_AREA])); largest=labels==component
    cx,cy=centroids[component]; center_x,center_y=(mask.shape[1]-1)/2,(mask.shape[0]-1)/2
    offset=math.hypot(cx-center_x,cy-center_y)/max(math.hypot(center_x,center_y),1)
    sides=int(largest[0,:].any())+int(largest[-1,:].any())+int(largest[:,0].any())+int(largest[:,-1].any())
    return {'lichen_fraction':fraction,'largest_component_fraction':float(largest.sum()/max(binary.sum(),1)),
            'centroid_offset':float(offset),'border_touch_sides':sides}

def assess_guided_capture(path:Path, t:CaptureThresholds=CaptureThresholds())->dict[str,Any]:
    coverage=predict_coverage(path); image=coverage['image']; mask=coverage['mask']
    metrics={**base_capture_metrics(image),**mask_framing_metrics(mask)}
    scores={'resolution':min(1.0,min(metrics['width'],metrics['height'])/t.min_short_side,
                             metrics['width']*metrics['height']/t.min_total_pixels),
            'sharpness':float(np.clip(metrics['blur_variance']/t.min_blur_variance,0,1)),
            'brightness':float(np.clip(1-abs(metrics['mean_luminance']-0.52)/0.42,0,1)),
            'contrast':float(np.clip(metrics['robust_contrast']/t.min_robust_contrast,0,1)),
            'exposure_clipping':float(np.clip(1-max(metrics['underexposed_fraction']/t.max_underexposed_fraction,
                                                     metrics['overexposed_fraction']/t.max_overexposed_fraction),0,1)),
            'lichen_presence':float(np.clip(metrics['lichen_fraction']/t.min_lichen_fraction,0,1)),
            'framing':float(np.clip(1-metrics['centroid_offset']/t.max_centroid_offset,0,1))}
    warnings=[]; instructions=[]
    if min(metrics['width'],metrics['height'])<t.min_short_side or metrics['width']*metrics['height']<t.min_total_pixels:
        warnings.append('IMAGE_TOO_SMALL'); instructions.append('Use full-resolution camera mode.')
    if metrics['blur_variance']<t.min_blur_variance:
        warnings.append('IMAGE_TOO_BLURRY'); instructions.append('Hold the phone steady, tap to focus, and retake.')
    if metrics['mean_luminance']<t.min_mean_luminance:
        warnings.append('IMAGE_TOO_DARK'); instructions.append('Add indirect light or move out of deep shade.')
    elif metrics['mean_luminance']>t.max_mean_luminance:
        warnings.append('IMAGE_TOO_BRIGHT'); instructions.append('Reduce glare and avoid direct flash or harsh sunlight.')
    if metrics['robust_contrast']<t.min_robust_contrast:
        warnings.append('LOW_CONTRAST'); instructions.append('Improve focus and use even side lighting.')
    if metrics['overexposed_fraction']>t.max_overexposed_fraction:
        warnings.append('EXCESSIVE_HIGHLIGHT_CLIPPING'); instructions.append('Tilt the camera to remove glare.')
    if metrics['lichen_fraction']<t.min_lichen_fraction:
        warnings.append('TOO_LITTLE_LICHEN'); instructions.append('Move closer and center a larger lichen-covered area.')
    elif metrics['lichen_fraction']>t.max_lichen_fraction:
        instructions.append('Pull back slightly so the lichen boundary and some bark context are visible.')
    if metrics['centroid_offset']>t.max_centroid_offset:
        instructions.append('Recenter the detected lichen inside the guide frame.')
    if metrics['border_touch_sides']>t.max_border_touch_sides:
        instructions.append('Pull back slightly; detected lichen is cut off on most sides.')
    weights={'resolution':.14,'sharpness':.24,'brightness':.18,'contrast':.16,'exposure_clipping':.13,
             'lichen_presence':.08,'framing':.07}
    overall=sum(weights[k]*scores[k] for k in scores)/sum(weights.values())
    decision='retake_required' if warnings else ('accepted' if overall>=.78 else ('accepted_with_caution' if overall>=.62 else 'retake_required'))
    if not instructions and decision=='accepted': instructions=['Capture quality passed. Keep this framing for future repeat scans.']
    return {'feature':'guided_capture','version':'capture_guidance_v1.0.1_corrected_preprocessing',
            'decision':decision,'overall_score':round(float(overall),4),'model_status':'coverage_model_connected',
            'coverage_threshold':COVERAGE_THRESHOLD,'coverage_percent':round(coverage['coverage_percent'],2),
            'metrics':{k:round(float(v),4) if isinstance(v,(float,np.floating)) else int(v) for k,v in metrics.items()},
            'component_scores':{k:round(float(v),4) for k,v in scores.items()},'warning_codes':warnings,
            'instructions':instructions,'_image':image,'_mask':mask}

guided=assess_guided_capture(GUIDED_CAPTURE_IMAGE)
guided_json=OUTPUT_DIR/'feature19_capture_assessment.json'
guided_overlay=OUTPUT_DIR/'feature19_capture_overlay.png'
guided_json.write_text(json.dumps({k:v for k,v in guided.items() if not k.startswith('_')},indent=2))
image=guided['_image'].astype(np.float32); green=np.zeros_like(image); green[...,1]=255
alpha=(.38*guided['_mask'])[...,None]; overlay=(image*(1-alpha)+green*alpha).clip(0,255).astype(np.uint8)
canvas=Image.fromarray(overlay); draw=ImageDraw.Draw(canvas,'RGBA'); w,h=canvas.size
color=(32,190,110,255) if guided['decision']=='accepted' else (239,78,78,255)
draw.rounded_rectangle((int(.12*w),int(.12*h),int(.88*w),int(.88*h)),radius=max(8,int(min(w,h)*.02)),
                       outline=color,width=max(4,int(min(w,h)*.008)))
canvas.save(guided_overlay)
display(canvas)
print(json.dumps({k:v for k,v in guided.items() if not k.startswith('_')},indent=2))
print('Saved:',guided_json,guided_overlay)

In [ ]:
# Cell 10 — Final readiness scorecard and files to keep
feature18_pass = alignment['decision'] in {'accepted','caution'}
feature19_pass = guided['model_status']=='coverage_model_connected' and guided['decision'] in {'accepted','accepted_with_caution'}
scorecard = {
    'feature_18': {
        'status':'PASS' if feature18_pass else 'RETAKE_REQUIRED',
        'coverage_model_ran_on_both_images':True,
        'manifest_created':MANIFEST_PATH.exists(),
        'comparability_decision':alignment['decision'],
        'change_status':coverage_change.get('change_status'),
        'capture_dates_known':bool(BASELINE_CAPTURED_AT and CURRENT_CAPTURED_AT),
        'elapsed_time_and_trend_claims_enabled':False,
    },
    'feature_19': {
        'status':'PASS' if feature19_pass else 'RETAKE_REQUIRED',
        'coverage_model_connected':True,
        'decision':guided['decision'],
        'warning_codes':guided['warning_codes'],
    },
    'required_outputs': {
        'repeat_manifest':str(MANIFEST_PATH),
        'repeat_result':str(repeat_path),
        'feature19_assessment':str(guided_json),
        'feature19_overlay':str(guided_overlay),
    },
}
scorecard_path=OUTPUT_DIR/'features_18_19_real_qa_scorecard.json'
scorecard_path.write_text(json.dumps(scorecard,indent=2))
print(json.dumps(scorecard,indent=2))
print('\nSaved final scorecard:',scorecard_path)

## How to interpret the result

- **Feature 18 PASS:** the photographs were comparable enough to calculate a screening-level change. `no_reliable_change_detected` is a scientifically valid result and is not a failure.
- **Feature 18 RETAKE_REQUIRED:** do not use the apparent coverage difference. Retake the newer photograph from the original distance, angle, zoom, and tree section.
- **Feature 19 PASS:** the coverage model was connected and the image was accepted or accepted with caution.
- **Feature 19 RETAKE_REQUIRED:** follow the ordered instructions and rerun Cells 3, 5, 7–10 after replacing the photograph.

When dates are unknown, the result is limited to an explicitly ordered older-versus-latest comparison. It cannot report elapsed time, annualized change, or a trend. If you later recover dates, add them to the configuration and rerun. Even with dates, two observations support a pairwise comparison—not a long-term trend. A trend requires at least three comparable dated observations spanning at least 30 days.